# 1. What you'll learn

        - train an autoencoder only on chosen normal digits
- calibrate an error threshold without anomaly labels
- show contamination teaching the model to reconstruct anomalies

        **The one question this notebook answers:** Can reconstruction error identify images unlike the normal-only training distribution?

# 2. Setup

This lesson keeps autoencoders for anomaly detection small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Digits 0–4 are treated as normal; digits 8–9 are anomalies for evaluation. Images are real 8×8 handwritten digits and labels never enter reconstruction training.

**Small example:** For five pixels truth [0,1,1,0,.5] and reconstruction [0,.9,.8,.1,.4], squared errors sum to .07; an unfamiliar shape might produce a much larger sum.

In [ ]:
d=load_digits();X=(d.data/16).astype("float32");labels=d.target;normal=X[labels<5];anomaly=X[labels>=8];train_n,test_n=train_test_split(normal,test_size=.3,random_state=SEED);test=np.vstack([test_n,anomaly]);truth=np.r_[np.zeros(len(test_n),int),np.ones(len(anomaly),int)];print("Normal train:",train_n.shape,"test mix:",test.shape,"anomaly share:",round(truth.mean(),3))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(np.hstack([train_n[i].reshape(8,8) for i in range(8)]),cmap="gray_r");axes[0].axis("off");axes[0].set_title("Normal digits 0–4")
axes[1].imshow(np.hstack([anomaly[i].reshape(8,8) for i in range(8)]),cmap="gray_r");axes[1].axis("off");axes[1].set_title("Held-out anomalies 8–9")
axes[2].bar(["normal test","anomaly"],[len(test_n),len(anomaly)],color=["#176b66","#d38b45"]);axes[2].set_title("Evaluation composition");plt.tight_layout();plt.show()

**Takeaway:** Normal training still contains varied handwriting.

**Takeaway:** Anomalies share many strokes with normal digits, making the task nontrivial.

**Takeaway:** The evaluation mixture is not a prevalence estimate; average precision is interpreted against its explicit baseline.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
loader=DataLoader(TensorDataset(torch.tensor(train_n)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED));cal=train_n[-150:];fit=train_n[:-150];loader=DataLoader(TensorDataset(torch.tensor(fit)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED));print("Fit/calibration/test:",fit.shape,cal.shape,test.shape)

# 6. Train

        Training turns the assumptions behind autoencoders for anomaly detection into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Compress each normal image through a narrow latent bottleneck.
2. Decode the latent code back to 64 pixels.
3. Minimize normal-only reconstruction squared error.
4. Set a high calibration-error quantile as the anomaly threshold.

In [ ]:
torch.manual_seed(SEED);ae=nn.Sequential(nn.Linear(64,24),nn.ReLU(),nn.Linear(24,5),nn.ReLU(),nn.Linear(5,24),nn.ReLU(),nn.Linear(24,64),nn.Sigmoid());opt=torch.optim.Adam(ae.parameters(),lr=.005);losses=[]
for epoch in range(22):
 total=0
 for (xb,) in loader: opt.zero_grad();loss=nn.functional.mse_loss(ae(xb),xb);loss.backward();opt.step();total+=loss.item()*len(xb)
 losses.append(total/len(fit))
with torch.no_grad(): cal_err=((ae(torch.tensor(cal))-torch.tensor(cal))**2).mean(1).numpy()
threshold=np.quantile(cal_err,.95);print("Threshold:",round(threshold,4),"loss first/last:",round(losses[0],4),round(losses[-1],4))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
with torch.no_grad(): recon=ae(torch.tensor(test));error=((recon-torch.tensor(test))**2).mean(1).numpy();ap=average_precision_score(truth,error);pred=error>=threshold;precision=(truth[pred].mean() if pred.any() else 0);baseline=truth.mean()
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(losses);axes[0].set_title("Normal-only reconstruction loss")
axes[1].hist(error[truth==0],bins=25,alpha=.6,label="normal");axes[1].hist(error[truth==1],bins=25,alpha=.6,label="anomaly");axes[1].axvline(threshold,ls="--",color="black");axes[1].legend();axes[1].set_title("Reconstruction error")
axes[2].imshow(np.hstack([test[0].reshape(8,8),recon[0].numpy().reshape(8,8),test[-1].reshape(8,8),recon[-1].numpy().reshape(8,8)]),cmap="gray_r");axes[2].axis("off");axes[2].set_title("Normal/recon | anomaly/recon");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"AP={ap:.3f} vs prevalence={baseline:.3f}; flagged precision={precision:.3f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Training contamination teaches the autoencoder to reconstruct anomaly patterns, shrinking the very score separation detection needs. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
contaminated=np.vstack([fit,anomaly[:300]]);bad=nn.Sequential(nn.Linear(64,24),nn.ReLU(),nn.Linear(24,5),nn.ReLU(),nn.Linear(5,24),nn.ReLU(),nn.Linear(24,64),nn.Sigmoid());bad.load_state_dict(ae.state_dict());o=torch.optim.Adam(bad.parameters(),lr=.005);bad_x=torch.tensor(contaminated)
for _ in range(18): o.zero_grad();loss=nn.functional.mse_loss(bad(bad_x),bad_x);loss.backward();o.step()
with torch.no_grad(): bad_error=((bad(torch.tensor(test))-torch.tensor(test))**2).mean(1).numpy()
print(f"Normal-only AP={ap:.3f}; contaminated-training AP={average_precision_score(truth,bad_error):.3f}")

**Use this when…** you have representative normal data and anomalies should be hard for a capacity-limited reconstruction model.

        **Don't use this when…** anomalies are common in training, the decoder generalizes too well, or costly false alarms lack labelled threshold data.

        ## Try this

        1. Change bottleneck from 2 to 32 and track AP.
2. Treat digit 5 alone as anomaly.
3. Calibrate threshold at 90%, 95%, and 99%.